We will note that to route messages through the internet, other algorithms are used for finding the shortest path. One of the problems with using Dijkstra’s algorithm on the internet is that you must have a complete representation of the graph in order for the algorithm to run. The implication of this is that every router has a complete map of all the routers in the internet. In practice this is not the case and other variations of the algorithm allow each router to discover the graph as they go. One such algorithm that you may want to read about is called the distance vector routing algorithm.



Other references:
- Dijkstras Shortest Path Algorithm Explained | With Example | Graph Theory FelixTechTips: https://youtu.be/bZkzH5x0SKU?si=fDvGle9ynq9jbMCz
- MIT OpenCourseWare Introduction to Algorithms Lecture 13: Dijkstra's Algorithm: https://ocw.mit.edu/courses/6-006-introduction-to-algorithms-spring-2020/resources/mit6_006s20_lec13/
- check network models as explained in this book: *Operations Research APPLICATIONS AND ALGORITHMS FOURTH EDITION Wayne L. Winston INDIANA UNIVERSITY*


#### 🛣️ Shortest Path Selection: Dijkstra's Algorithm Logic

Dijkstra’s algorithm identifies the single-source shortest path across a weighted graph containing **non-negative weights**. It functions as an optimized level-order exploration engine that updates routing decisions on the fly.

##### 1. The Core Update Evaluation Strategy
As the search moves from an active node \(u\) to check an adjacent neighbor node \(v\), it evaluates an optimization inequality known as **Edge Relaxation**:

\[\text{if } \text{distance}(u) + \text{weight}(u, v) < \text{distance}(v)\]

If this assertion resolves to `True`, it indicates that traveling through node \(u\) yields a shorter route than any previous configuration encountered. The algorithm instantly overwrites `distance(v)` with the lower cost and maps `v.previous = u` to re-anchor the optimal pathway links.

##### 2. The Priority Queue Pipeline Efficiency
The algorithm leverages a **Priority Queue** using accumulated distance values as key weights. This ensures that the next element pulled out for exploration is *always* the unvisited node sitting at the absolute closest distance.

By utilizing the custom **`change_priority`** method:
* The minute a node’s distance is relaxed and reduced, its position inside the binary min-heap updates instantly.
* The vertex bubbles up toward the front of the tracking array, allowing it to be audited ahead of more distant, less optimal routing options.
* This optimization cuts down search paths, ensuring that once a vertex is dequeued from the priority queue, its absolute shortest distance from the origin has been completely resolved.


In [3]:
import sys

# =====================================================================
# 1. GRAPH BASE MODULES WITH DIJKSTRA WEIGHT TRACKING
# =====================================================================

class Vertex:
    def __init__(self, key):
        """Initializes a vertex tracking weights and routing predecessors."""
        self.id = key
        self.neighbors = {}  # Format: {neighbor_vertex_object: edge_weight}
        
        # Dijkstra path computation attributes
        self.distance = sys.maxsize  # Initialised to infinity (large max integer)
        self.previous = None         # Holds pointer to the routing predecessor node

    def set_neighbor(self, nbr, weight=0):
        """Adds a weighted connection to a neighbor node."""
        self.neighbors[nbr] = weight

    def get_neighbors(self):
        """Returns all adjacent vertex objects."""
        return self.neighbors.keys()

    def get_key(self):
        return self.id

    def get_weight(self, nbr):
        """Returns the weight cost to reach the target neighbor vertex."""
        return self.neighbors[nbr]

    def __repr__(self):
        return f"Vertex({self.id}, dist={self.distance if self.distance != sys.maxsize else 'inf'})"


class Graph:
    def __init__(self):
        self.vertices = {}

    def set_vertex(self, key):
        if key not in self.vertices:
            self.vertices[key] = Vertex(key)
        return self.vertices[key]

    def get_vertex(self, key):
        return self.vertices.get(key, None)

    def add_edge(self, from_key, to_key, weight=0):
        """Creates a directed edge with a specified numerical cost weight."""
        if from_key not in self.vertices:
            self.set_vertex(from_key)
        if to_key not in self.vertices:
            self.set_vertex(to_key)
        self.vertices[from_key].set_neighbor(self.vertices[to_key], weight)

    def __iter__(self):
        return iter(self.vertices.values())


# =====================================================================
# 2. PRIORITY QUEUE ADT (Binary Min-Heap with decrease-key capacity)
# =====================================================================

class PriorityQueue:
    def __init__(self):
        # 1-indexed heap list storing tuples: [0, (priority, key), ...]
        self.heap_list = [0]
        self.current_size = 0

    def _perc_up(self, i):
        """Bubbles up a node to preserve the min-heap configuration structure."""
        while i // 2 > 0:
            if self.heap_list[i][0] < self.heap_list[i // 2][0]:
                self.heap_list[i], self.heap_list[i // 2] = self.heap_list[i // 2], self.heap_list[i]
            i = i // 2

    def _perc_down(self, i):
        """Pushes down a node to preserve the min-heap configuration structure."""
        while (i * 2) <= self.current_size:
            mc = self._min_child(i)
            if self.heap_list[i][0] > self.heap_list[mc][0]:
                self.heap_list[i], self.heap_list[mc] = self.heap_list[mc], self.heap_list[i]
            i = mc

    def _min_child(self, i):
        if i * 2 + 1 > self.current_size:
            return i * 2
        else:
            if self.heap_list[i * 2][0] < self.heap_list[i * 2 + 1][0]:
                return i * 2
            return i * 2 + 1

    def enqueue(self, item_tuple):
        """Adds a new (priority, key) tuple element into the queue."""
        self.heap_list.append(item_tuple)
        self.current_size += 1
        self._perc_up(self.current_size)

    def dequeue_min(self):
        """Pops and returns the item tuple holding the smallest active priority."""
        ret_val = self.heap_list[1]
        self.heap_list[1] = self.heap_list[self.current_size]
        self.current_size -= 1
        self.heap_list.pop()
        self._perc_down(1)
        return ret_val

    def change_priority(self, key, new_priority):
        """
        Locates the key entry inside the heap structure and updates its priority,
        percolating it up toward the front of the queue seamlessly.
        """
        # Linear scan to locate the index tracking the target key
        for idx in range(1, self.current_size + 1):
            if self.heap_list[idx][1] == key:
                # Update the priority value (distance) inline
                self.heap_list[idx] = (new_priority, key)
                # Percolate up since the weight distance is only reduced in Dijkstra
                self._perc_up(idx)
                break

    def is_empty(self):
        return self.current_size == 0

    def build_queue(self, initial_tuples):
        """Builds a batch priority queue structure in linear time."""
        self.current_size = len(initial_tuples)
        self.heap_list = [0] + list(initial_tuples)
        i = self.current_size // 2
        while i > 0:
            self._perc_down(i)
            i -= 1


# =====================================================================
# 3. DIJKSTRA'S SHORTEST PATH ROUTING ENGINE
# =====================================================================

def dijkstra(graph, start_vertex):
    """
    Listing 7.20.1: Iterative Dijkstra algorithm calculating optimal path costs
    from a specified starting vertex to all other accessible nodes.
    """
    # 1. Set the initial distance of the origin start vertex node to 0 [1]
    start_vertex.distance = 0
    
    # 2. Assemble and build the priority queue loaded with all graph vertices [1]
    pq = PriorityQueue()
    initial_elements = [(v.distance, v.get_key()) for v in graph]
    pq.build_queue(initial_elements)

    while not pq.is_empty():
        # Dequeue the vertex with the absolute smallest accumulated distance [1]
        current_dist, current_key = pq.dequeue_min()
        current_vertex = graph.get_vertex(current_key)
        
        # Systematically explore each neighbor attached to this node
        for next_vertex in current_vertex.get_neighbors():
            # Calculate the total path weight if traveling through the current node
            new_distance = current_vertex.distance + current_vertex.get_weight(next_vertex)
            
            # If this alternative pathway is cheaper than the node's prior known distance [1]
            if new_distance < next_vertex.distance:
                # Update distance metrics and reset predecessor routing pointers [1]
                next_vertex.distance = new_distance
                next_vertex.previous = current_vertex
                
                # Signal the priority queue to pull this vertex toward the front [1]
                pq.change_priority(next_vertex.get_key(), new_distance)


def print_shortest_path(graph, end_key):
    """Traces predecessor pointers backwards to reveal the final routing path layout."""
    end_vertex = graph.get_vertex(end_key)
    path = []
    current = end_vertex
    while current is not None:
        path.append(current.get_key())
        current = current.previous
    path.reverse()
    print(f"🛣️  Shortest Path to {end_key:<2} : {' -> '.join(path):<15} (Total Cost: {end_vertex.distance})")


# =====================================================================
# 4. VERIFICATION SYSTEM DRIVER (Graph Mirroring Figures 3-8)
# =====================================================================

# Build the custom graph featured in your textbook example trace text [1]
g = Graph()

g.add_edge("u", "v", 2)
g.add_edge("u", "w", 5)
g.add_edge("u", "x", 1)

g.add_edge("v", "u", 2)
g.add_edge("v", "w", 3)
g.add_edge("v", "x", 2)

g.add_edge("x", "u", 1)
g.add_edge("x", "v", 2)
g.add_edge("x", "w", 3)
g.add_edge("x", "y", 1)

g.add_edge("w", "u", 5)
g.add_edge("w", "v", 3)
g.add_edge("w", "x", 3)
g.add_edge("w", "y", 1)
g.add_edge("w", "z", 5)

g.add_edge("y", "x", 1)
g.add_edge("y", "w", 1)
g.add_edge("y", "z", 1)

g.add_edge("z", "w", 5)
g.add_edge("z", "y", 1)


print("=====================================================================")
print("⚡ RUNNING DIJKSTRA SHORTED PATHS SCHEDULER ENGINE")
print("=====================================================================\n")

# Run calculation starting at node 'u' [1]
start_node = g.get_vertex("u")
dijkstra(g, start_node)

print("🏁 OPTIMAL ROUTING MAP RESOLUTION MATRIX:")
print("-" * 55)
for key in sorted(g.vertices.keys()):
    if key != "u":
        print_shortest_path(g, key)
print("-" * 55)


⚡ RUNNING DIJKSTRA SHORTED PATHS SCHEDULER ENGINE

🏁 OPTIMAL ROUTING MAP RESOLUTION MATRIX:
-------------------------------------------------------
🛣️  Shortest Path to v  : u -> v          (Total Cost: 2)
🛣️  Shortest Path to w  : u -> x -> y -> w (Total Cost: 3)
🛣️  Shortest Path to x  : u -> x          (Total Cost: 1)
🛣️  Shortest Path to y  : u -> x -> y     (Total Cost: 2)
🛣️  Shortest Path to z  : u -> x -> y -> z (Total Cost: 3)
-------------------------------------------------------


In [5]:
import sys

"""the priority queue in Dijkstra's algorithm needs to look up
    a vertex by its string key name alone when determining membership,
    rather than hunting for a strict weight matching tuple.
    Adding the custom __contains__ magic method completely prevents
    the graph entries from getting stuck at infinity..
"""


# =====================================================================
# 1. GRAPH BASE MODULES WITH WEIGHT TRACKING
# =====================================================================

class Vertex:
    def __init__(self, key):
        """Initializes a vertex tracking weights and routing predecessors."""
        self.id = key
        self.neighbors = {}  # Format: {neighbor_vertex_object: edge_weight}
        self.distance = sys.maxsize  # Initialized to infinity
        self.previous = None         # Predecessor parent node

    def set_neighbor(self, nbr, weight=0):
        self.neighbors[nbr] = weight

    def get_neighbors(self):
        return self.neighbors.keys()

    def get_key(self):
        return self.id

    def get_weight(self, nbr):
        return self.neighbors[nbr]

    def __repr__(self):
        return f"Vertex({self.id}, dist={self.distance if self.distance != sys.maxsize else 'inf'})"


class Graph:
    def __init__(self):
        self.vertices = {}

    def set_vertex(self, key):
        if key not in self.vertices:
            self.vertices[key] = Vertex(key)
        return self.vertices[key]

    def get_vertex(self, key):
        return self.vertices.get(key, None)

    def add_edge(self, from_key, to_key, weight=0):
        if from_key not in self.vertices:
            self.set_vertex(from_key)
        if to_key not in self.vertices:
            self.set_vertex(to_key)
        self.vertices[from_key].set_neighbor(self.vertices[to_key], weight)

    def __iter__(self):
        return iter(self.vertices.values())


# =====================================================================
# 2. PRIORITY QUEUE ADT (With Key Membership Check Fix)
# =====================================================================

class PriorityQueue:
    def __init__(self):
        self.heap_list = [0]  # 1-indexed placeholder
        self.current_size = 0

    def _perc_up(self, i):
        while i // 2 > 0:
            if self.heap_list[i] < self.heap_list[i // 2]:
                self.heap_list[i], self.heap_list[i // 2] = self.heap_list[i // 2], self.heap_list[i]
            i = i // 2

    def _perc_down(self, i):
        while (i * 2) <= self.current_size:
            mc = self._min_child(i)
            if self.heap_list[i] > self.heap_list[mc]:
                self.heap_list[i], self.heap_list[mc] = self.heap_list[mc], self.heap_list[i]
            i = mc

    def _min_child(self, i):
        if i * 2 + 1 > self.current_size:
            return i * 2
        else:
            if self.heap_list[i * 2] < self.heap_list[i * 2 + 1]:
                return i * 2
            return i * 2 + 1

    def dequeue_min(self):
        ret_val = self.heap_list[1]
        self.heap_list[1] = self.heap_list[self.current_size]
        self.current_size -= 1
        self.heap_list.pop()
        self._perc_down(1)
        return ret_val

    def change_priority(self, key, new_priority):
        """Locates the entry index matching the string key and reduces its weight."""
        for idx in range(1, self.current_size + 1):
            if self.heap_list[idx][1] == key:
                self.heap_list[idx] = (new_priority, key)
                self._perc_up(idx)
                break

    def __contains__(self, key):
        """
        ✨ membership verification fix: Safely checks if a vertex name string 
        is still in the queue by matching it against the key tuple element.
        """
        for idx in range(1, self.current_size + 1):
            if self.heap_list[idx][1] == key:
                return True
        return False

    def is_empty(self):
        return self.current_size == 0

    def build_queue(self, initial_tuples):
        self.current_size = len(initial_tuples)
        self.heap_list = [0] + list(initial_tuples)
        i = self.current_size // 2
        while i > 0:
            self._perc_down(i)
            i -= 1


# =====================================================================
# 3. DIJKSTRA'S SHORTED PATH ENGINE
# =====================================================================

def dijkstra(graph, start_vertex):
    start_vertex.distance = 0
    pq = PriorityQueue()
    initial_elements = [(v.distance, v.get_key()) for v in graph]
    pq.build_queue(initial_elements)

    while not pq.is_empty():
        current_dist, current_key = pq.dequeue_min()
        current_vertex = graph.get_vertex(current_key)
        
        for next_vertex in current_vertex.get_neighbors():
            # Dijkstra rule: accumulate the direct edge cost onto the current path distance
            new_distance = current_vertex.distance + current_vertex.get_weight(next_vertex)
            
            # Use our custom string membership check safely
            if next_vertex.get_key() in pq and new_distance < next_vertex.distance:
                next_vertex.distance = new_distance
                next_vertex.previous = current_vertex
                pq.change_priority(next_vertex.get_key(), new_distance)


def print_shortest_path(graph, end_key):
    end_vertex = graph.get_vertex(end_key)
    path = []
    current = end_vertex
    while current is not None:
        path.append(current.get_key())
        current = current.previous
    path.reverse()
    print(f"🛣️  Shortest Path to {end_key:<2} : {' -> '.join(path):<15} (Total Cost: {end_vertex.distance})")


# =====================================================================
# 4. VERIFICATION SYSTEM DRIVER (Graph Mirroring Figures 3-8)
# =====================================================================

g = Graph()

g.add_edge("u", "v", 2)
g.add_edge("u", "w", 5)
g.add_edge("u", "x", 1)

g.add_edge("v", "u", 2)
g.add_edge("v", "w", 3)
g.add_edge("v", "x", 2)

g.add_edge("x", "u", 1)
g.add_edge("x", "v", 2)
g.add_edge("x", "w", 3)
g.add_edge("x", "y", 1)

g.add_edge("w", "u", 5)
g.add_edge("w", "v", 3)
g.add_edge("w", "x", 3)
g.add_edge("w", "y", 1)
g.add_edge("w", "z", 5)

g.add_edge("y", "x", 1)
g.add_edge("y", "w", 1)
g.add_edge("y", "z", 1)

g.add_edge("z", "w", 5)
g.add_edge("z", "y", 1)


print("=====================================================================")
print("⚡ RUNNING DIJKSTRA SHORTED PATHS SCHEDULER ENGINE")
print("=====================================================================\n")

start_node = g.get_vertex("u")
dijkstra(g, start_node)

print("🏁 OPTIMAL ROUTING MAP RESOLUTION MATRIX:")
print("-" * 55)
for key in sorted(g.vertices.keys()):
    if key != "u":
        print_shortest_path(g, key)
print("-" * 55)


⚡ RUNNING DIJKSTRA SHORTED PATHS SCHEDULER ENGINE

🏁 OPTIMAL ROUTING MAP RESOLUTION MATRIX:
-------------------------------------------------------
🛣️  Shortest Path to v  : u -> v          (Total Cost: 2)
🛣️  Shortest Path to w  : u -> x -> y -> w (Total Cost: 3)
🛣️  Shortest Path to x  : u -> x          (Total Cost: 1)
🛣️  Shortest Path to y  : u -> x -> y     (Total Cost: 2)
🛣️  Shortest Path to z  : u -> x -> y -> z (Total Cost: 3)
-------------------------------------------------------


#### 📡 Decentralized Routing: The Distance Vector Algorithm

The **Distance Vector Routing Algorithm** is a decentralized, iterative protocol used by physical network hardware routers to dynamically build global routing maps without any single machine requiring a complete view of the network topology.

##### 1. Mathematical Mechanics: The Bellman-Ford Equation
Each independent node maintains a localized vector array tracking its lowest known cost to all other addresses across the grid ecosystem. Periodically, routers duplicate their metrics table and transmit it straight to their direct neighbors. 

When a router receives an update from a neighbor v, it recalculates its own options using the **Bellman-Ford equation**:

\[D_x(y) = \min_v \{c(x,v) + D_v(y)\}\]

Where:
* c(x,v) is the direct, local connection cost from router x to its neighbor v.
* \(D_v(y)\) is neighbor v's reported cost to get from itself to the ultimate target destination y.

If the combination path through neighbor v is cheaper than the router's current stored record, the router instantly rewrites its internal routing table entry with the new lower weight and updates its `Next Hop` parameter to point to v.

##### 2. Key Differences: Centralized vs. Decentralized Routing

| Architectural Feature | Centralized Link-State (Dijkstra) | Distributed Distance Vector (Bellman-Ford) |
| :--- | :--- | :--- |
| **Topology View** | Every single router must possess an identical, complete map of the entire graph layout before running. | Each router only knows its direct neighbors and relies on indirect gossip metrics for distant paths. |
| **Computation Style** | Calculated locally in a single pass using a fast priority queue container array structure. | Calculated collectively and asynchronously through iterative packet updates. |
| **Network Scale Overhead** | High memory allocation overhead to store exceptionally massive graph structures on every chip. | Low storage overhead; each node only keeps an array proportional to the total number of destinations. |
| **Convergence Risk** | Susceptible to global performance bottlenecks if link states change rapidly. | Susceptible to routing loops and slow convergence issues (known as the *"Count-to-Infinity"* problem). |


In [2]:
import sys
import copy

class Router:
    def __init__(self, name):
        """Initializes a router node with a local routing table and known neighbors."""
        self.name = name
        # Links stores direct neighbor costs: {neighbor_name: edge_weight}
        self.links = {}
        
        # Distance Vector Matrix Table: {destination_node: (total_cost, next_hop_node)}
        self.distance_vector = {}
        
        # Base setup: A router can reach itself at 0 cost with no next hop
        self.distance_vector[self.name] = (0, self.name)

    def add_direct_link(self, neighbor_name, cost):
        """Registers a physical direct link connection cost."""
        self.links[neighbor_name] = cost
        # Initialize our table with this direct link info
        self.distance_vector[neighbor_name] = (cost, neighbor_name)

    def initialize_unknown_nodes(self, all_nodes):
        """Initializes unlinked nodes to infinity cost."""
        for node in all_nodes:
            if node not in self.distance_vector:
                self.distance_vector[node] = (sys.maxsize, None)

    def update_distance_vector(self, neighbor_name, neighbor_vector):
        """
        Core Bellman-Ford Engine Step:
        Evaluates a neighbor's vector table copy to see if routing through them
        lowers the cost to reach any destination node across the network map.
        """
        updated = False
        cost_to_neighbor = self.links[neighbor_name]

        for destination, (nb_cost, _) in neighbor_vector.items():
            # If the neighbor cannot reach the destination, skip evaluating it
            if nb_cost == sys.maxsize:
                continue

            # Calculate total cost if we route through this neighbor
            new_cost = cost_to_neighbor + nb_cost
            current_cost = self.distance_vector[destination][0]

            # If this route is cheaper, update our distance vector table inline
            if new_cost < current_cost:
                self.distance_vector[destination] = (new_cost, neighbor_name)
                updated = True
                
        return updated

    def print_routing_table(self):
        """Prints a grid layout matrix of this specific router's active table choices."""
        print(f"\n📊 Routing Table for Router [{self.name}]:")
        print("-" * 45)
        print(f"{'Destination':<15} | {'Shortest Cost':<13} | {'Next Hop':<10}")
        print("-" * 45)
        for dest in sorted(self.distance_vector.keys()):
            cost, next_hop = self.distance_vector[dest]
            cost_str = str(cost) if cost != sys.maxsize else "inf"
            hop_str = str(next_hop) if next_hop else "None"
            print(f"{dest:<15} | {cost_str:<13} | {hop_str:<10}")
        print("-" * 45)


class NetworkSimulation:
    def __init__(self):
        self.routers = {}

    def add_router(self, name):
        self.routers[name] = Router(name)

    def connect_routers(self, r1, r2, cost):
        """Creates a bidirectional physical network wire link between two routers."""
        self.routers[r1].add_direct_link(r2, cost)
        self.routers[r2].add_direct_link(r1, cost)

    def run_routing_updates(self):
        """
        Simulates iterative convergence rounds. Routers pass copies of their 
        vectors to neighbors until no more table updates occur.
        """
        all_node_names = list(self.routers.keys())
        for router in self.routers.values():
            router.initialize_unknown_nodes(all_node_names)

        converged = False
        round_num = 1

        while not converged:
            print(f"🔄 --- Network Convergence Optimization Pass Round #{round_num} ---")
            any_changes_this_round = False

            # Create a frozen snapshot copy of all vectors at the start of this round
            # to simulate independent concurrent message packet passes
            snapshot_vectors = {name: copy.deepcopy(r.distance_vector) for name, r in self.routers.items()}

            # For every router, share its neighbors' information with it
            for router_name, router in self.routers.items():
                for neighbor_name in router.links:
                    # Pass the neighbor's vector snapshot to the current router
                    neighbor_data = snapshot_vectors[neighbor_name]
                    changed = router.update_distance_vector(neighbor_name, neighbor_data)
                    if changed:
                        any_changes_this_round = True

            if not any_changes_this_round:
                print("✨ Network has fully converged! No more table adjustments occurred.")
                converged = True
            else:
                round_num += 1
            print()


# =====================================================================
# 🚀 Running the Network Simulation
# =====================================================================

sim = NetworkSimulation()

# Create 4 independent routers acting as a distributed topology
sim.add_router("Router_A")
sim.add_router("Router_B")
sim.add_router("Router_C")
sim.add_router("Router_D")

# Link connections matching a mesh layout:
#  A ──(2)── B
#  │         │
# (7)       (1)
#  │         │
#  C ──(3)── D
sim.connect_routers("Router_A", "Router_B", 2)
sim.connect_routers("Router_A", "Router_C", 7)
sim.connect_routers("Router_B", "Router_D", 1)
sim.connect_routers("Router_C", "Router_D", 3)

# Execute the iterative table exchange loops
sim.run_routing_updates()

print("\n" + "=" * 55)
print("🏁 FINAL DISTRIBUTED ROUTING RESOLUTIONS MATRIX")
print("=" * 55)
# Print final decentralized results
sim.routers["Router_A"].print_routing_table()
sim.routers["Router_C"].print_routing_table()


🔄 --- Network Convergence Optimization Pass Round #1 ---

🔄 --- Network Convergence Optimization Pass Round #2 ---

🔄 --- Network Convergence Optimization Pass Round #3 ---
✨ Network has fully converged! No more table adjustments occurred.


🏁 FINAL DISTRIBUTED ROUTING RESOLUTIONS MATRIX

📊 Routing Table for Router [Router_A]:
---------------------------------------------
Destination     | Shortest Cost | Next Hop  
---------------------------------------------
Router_A        | 0             | Router_A  
Router_B        | 2             | Router_B  
Router_C        | 6             | Router_B  
Router_D        | 3             | Router_B  
---------------------------------------------

📊 Routing Table for Router [Router_C]:
---------------------------------------------
Destination     | Shortest Cost | Next Hop  
---------------------------------------------
Router_A        | 6             | Router_D  
Router_B        | 4             | Router_D  
Router_C        | 0             | Router_